# Example 10.2 / 1: Solve a Transportation Problem

Companion notebook for **10.2 Transportation and Assignment Networks** in the AMPL lab Documentation, section "Solve a Transportation Problem". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a Transportation Problem

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIGINS;  # supply points
set DESTINATIONS;  # demand points
param supply {ORIGINS} >= 0;  # units available at each origin
param demand {DESTINATIONS} >= 0;  # units required at each destination
param unit_cost {ORIGINS, DESTINATIONS} >= 0;  # cost per unit shipped

# balanced problem: supply equals demand
check: sum {i in ORIGINS} supply[i] = sum {j in DESTINATIONS} demand[j];

var Ship {ORIGINS, DESTINATIONS} >= 0;  # units sent from i to j

minimize TotalCost:  # total transportation cost
    sum {i in ORIGINS, j in DESTINATIONS} unit_cost[i, j] * Ship[i, j];

subject to SupplyUsed {i in ORIGINS}:  # each origin ships all its supply
    sum {j in DESTINATIONS} Ship[i, j] = supply[i];

subject to DemandMet {j in DESTINATIONS}:  # each destination receives its demand
    sum {i in ORIGINS} Ship[i, j] = demand[j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ORIGINS"] = ["north", "south"]
ampl.set["DESTINATIONS"] = ["retail", "online"]
ampl.param["supply"] = {"north": 4, "south": 6}
ampl.param["demand"] = {"retail": 5, "online": 5}
ampl.param["unit_cost"] = {("north", "retail"): 2, ("north", "online"): 4, ("south", "retail"): 3, ("south", "online"): 1}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Ship"].to_dict())

HiGHS 1.14.0: optimal solution; objective 16
0 simplex iterations
0 barrier iterations
solved
{('north', 'online'): 0, ('north', 'retail'): 4, ('south', 'online'): 5, ('south', 'retail'): 1}


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [6]:
%%writefile data.dat
set ORIGINS := north south;
set DESTINATIONS := retail online;
param supply := north 4 south 6;
param demand := retail 5 online 5;
param unit_cost: retail online := north 2 4 south 3 1;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Ship;

HiGHS 1.14.0: optimal solution; objective 16
0 simplex iterations
0 barrier iterations
Ship :=
north online   0
north retail   4
south online   5
south retail   1
;

